# Experiment: Team managed experiment

Этот template копируется командой `make new-experiment`. Вставьте код обучения в одну функцию и нажмите **Run All** — остальное обрабатывается автоматически.

In [ ]:
# Setup: найдите корень клонированного репозитория и импортируйте runner.
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "configs/project.json").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Откройте notebook внутри клонированного репозитория.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
if importlib.util.find_spec("clearml") is None:
    print("WARNING: установите ClearML один раз: pip install -e '.[tracking]'")
from pmldl_llm import (
    ExperimentOutput,
    load_experiment_setup,
    run_notebook_experiment,
)

PROJECT_ROOT

## 1. Готовый конфиг эксперимента

ID, owner, parent, seed, split и ClearML уже настроены командой `make new-experiment`. Эту ячейку менять не нужно.

In [ ]:
SETUP = load_experiment_setup("__EXPERIMENT_CONFIG__", project_root=PROJECT_ROOT)
SETUP

## 2. Ваш эксперимент

Перенесите существующий код обучения и оценки внутрь функции. В конце верните итоговые validation-метрики и пути к файлам, которые нужно сохранить.

In [ ]:
def train_and_evaluate(run):
    # ===== ВСТАВЬТЕ СЮДА СВОЙ КОД ОБУЧЕНИЯ =====
    # Пример live-логирования по эпохам:
    # run.log_metric("loss", train_loss, namespace="train", step=epoch)
    #
    # В результате вашего кода должны появиться числовые значения ниже
    # и, при необходимости, сохранённые model/predictions файлы.

    artifacts = {
        # Имя внутри run: путь к уже сохранённому файлу.
        # "models/model.pt": MODEL_PATH,
        # "predictions/validation.csv": PREDICTIONS_PATH,
    }
    # Все обязательные метрики и A/B averaging считаются автоматически.
    return ExperimentOutput.from_predictions(
        y_true=validation_targets,
        original_probabilities=validation_probabilities,
        swapped_back_probabilities=swapped_back_probabilities,
        artifacts=artifacts,
    )

## 3. Автоматический запуск

Эту ячейку менять не нужно. При ошибке run автоматически сохранится со статусом `failed`; при успехе он будет проверен и попадёт в ClearML, а full-run — в локальный leaderboard.

In [ ]:
RESULT = run_notebook_experiment(
    train_and_evaluate,
    SETUP,
    project_root=PROJECT_ROOT,
)
RESULT

## Что получится

- `configs/experiments/<experiment_id>.json` — созданный конфиг;
- `results/runs/<run_id>/` — проверенные метрики и metadata;
- `artifacts/<run_id>/` — модели и predictions;
- ClearML task — live-метрики;
- `results/leaderboard.csv` — автоматически обновлённое локальное сравнение.

Первый **Run All** запускается автоматически командой `make run-experiment EXPERIMENT=<ID>` в smoke-режиме. Если smoke успешен, команда сама переключит конфиг, проверит проект и выполнит полный запуск. После успеха выполните `make submit-experiment EXPERIMENT=<ID>`.